# 01-2. 行列 — 動かして確かめる

📖 解説: [`../02_matrices.md`](../02_matrices.md)

## このノートで触るもの
1. 行列の作成と shape
2. 特別な行列 (ゼロ・単位・対角・対称)
3. 転置
4. 行列×ベクトル（線形変換）
5. 行列×行列
6. 逆行列・連立方程式
7. 行列式とランク
8. ニューラルネット1層 + バッチ処理

> 🧭 **クイックナビ**: 📚 [ROOT (全体 TOP)](../../README.md) ・ 🏠 [章 TOP](../README.md) ・ 📖 [解説 md (02_matrices.md)](../02_matrices.md)

In [ ]:
import math
import numpy as np
import jax.numpy as jnp
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore", message=".*distutils Version classes.*", category=DeprecationWarning)
import japanize_matplotlib  # noqa: F401  # 日本語フォント (豆腐化対策)
from ipywidgets import interact, FloatSlider

%matplotlib inline
rng = np.random.default_rng(seed=42)

## 1. 行列の作成

$$
A \in \mathbb{R}^{m \times n}, \qquad A_{ij} \text{ は } i \text{ 行 } j \text{ 列の要素}
$$

In [ ]:
A = np.array([[1, 2, 3],
              [4, 5, 6]])
print(A)
print(f'shape: {A.shape}    ← (2, 3) は 2行3列')
print(f'ndim:  {A.ndim}     ← 2次元配列')
print(f'A[0, 1] = {A[0, 1]}    ← 0行目, 1列目 (数学では A[1, 2])')
print(f'A[0, :] = {A[0, :]}    ← 0行目すべて')
print(f'A[:, 1] = {A[:, 1]}    ← 1列目すべて')

## 2. 特別な行列

In [ ]:
print('ゼロ行列:')
print(np.zeros((2, 3)))
print('\n単位行列 I₃:')
print(np.eye(3))
print('\n対角行列:')
print(np.diag([2, 3, 5]))

## 3. 転置

$$
(A^\top)_{ij} = A_{ji}, \quad (AB)^\top = B^\top A^\top
$$

In [ ]:
A = np.array([[1, 2, 3],
              [4, 5, 6]])
print(f'A shape: {A.shape}')
print(f'A:\n{A}')
print(f'\nA.T shape: {A.T.shape}')
print(f'A.T:\n{A.T}')

# AᵀA は対称行列になる (機械学習頻出)
AtA = A.T @ A
print(f'\nAᵀA shape: {AtA.shape}    ← 必ず正方行列')
print(f'AᵀA:\n{AtA}')
print(f'対称か: {np.allclose(AtA, AtA.T)}')

## 4. 行列×ベクトル — 線形変換を可視化

回転行列 R を使って、ベクトル x を回転させてみます。

$$
R(\theta) = \begin{pmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{pmatrix}, \quad R(\theta) \mathbf{x} = \text{回転後}
$$

In [ ]:
def plot_rotation(theta_deg: float) -> None:
    """回転行列によるベクトル変換を可視化."""
    theta = math.radians(theta_deg)
    R = np.array([[math.cos(theta), -math.sin(theta)],
                  [math.sin(theta),  math.cos(theta)]])

    x = np.array([3.0, 1.0])    # 元のベクトル
    y = R @ x                    # 変換後

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.quiver(0, 0, x[0], x[1], angles='xy', scale_units='xy', scale=1, color='blue', label=f'x = {tuple(x)}')
    ax.quiver(0, 0, y[0], y[1], angles='xy', scale_units='xy', scale=1, color='red', label=f'Rx = ({y[0]:.2f}, {y[1]:.2f})')
    ax.set_xlim(-4, 4)
    ax.set_ylim(-4, 4)
    ax.axhline(0, color='black', linewidth=0.5)
    ax.axvline(0, color='black', linewidth=0.5)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=11)
    ax.set_title(f'回転変換 θ = {theta_deg}°')
    plt.show()

interact(
    plot_rotation,
    theta_deg=FloatSlider(min=0, max=360, step=10, value=45, description='角度 (度)'),
);

## 5. 行列×行列 — shape の確認が大事

$$
(AB)_{ij} = \sum_{k} A_{ik} B_{kj}
$$

shape: $(m, n) \times (n, p) \to (m, p)$

In [ ]:
A = np.array([[1, 2, 3],
              [4, 5, 6]])         # (2, 3)
B = np.array([[7,  8],
              [9,  10],
              [11, 12]])           # (3, 2)

C = A @ B
print(f'A.shape = {A.shape}, B.shape = {B.shape}')
print(f'A @ B.shape = {C.shape}    ← 「内側が一致、外側が結果」')
print(f'C =\n{C}')

# 順番を変えると...
C2 = B @ A
print(f'\nB @ A.shape = {C2.shape}    ← AB ≠ BA')

In [ ]:
# shape ミスマッチのエラー例 (実行するとエラー)
try:
    A = np.zeros((2, 3))
    B = np.zeros((4, 5))
    A @ B
except ValueError as e:
    print(f'ValueError: {e}')
    print('→ shape ミスマッチは最頻出のバグ。常に shape を print する習慣を')

## 6. 逆行列と連立方程式

```
2x +  y = 5
 x + 3y = 10
```
を `Ax = b` の形にして解きます。

$$
A A^{-1} = A^{-1} A = I
$$

連立方程式: $A\mathbf{x} = \mathbf{b} \implies \mathbf{x} = A^{-1}\mathbf{b}$

In [ ]:
A = np.array([[2.0, 1.0],
              [1.0, 3.0]])
b = np.array([5.0, 10.0])

# 方法1: 逆行列を計算 (推奨されない)
x1 = np.linalg.inv(A) @ b
print(f'方法1 (逆行列):  x = {x1}')

# 方法2: solve を使う (推奨)
x2 = np.linalg.solve(A, b)
print(f'方法2 (solve):   x = {x2}')

# 検算
print(f'\n検算 A @ x − b = {A @ x2 - b}    ← 0 に近ければOK')

## 7. 行列式とランク

$\det(A)$ = 線形変換の体積拡大率

$\det(A) = 0 \iff A$ は特異 (逆行列なし)

In [ ]:
A = np.array([[1.0, 2.0],
              [3.0, 4.0]])
print(f'det(A) = {np.linalg.det(A):.4f}    ← 1×4 − 2×3 = -2')
print(f'rank(A) = {np.linalg.matrix_rank(A)}')

# 特異な行列 (行が線形従属)
B = np.array([[1.0, 2.0],
              [2.0, 4.0]])         # 2行目 = 1行目 × 2
print(f'\ndet(B) = {np.linalg.det(B):.4f}    ← 0 = 特異')
print(f'rank(B) = {np.linalg.matrix_rank(B)}    ← 1 (情報が1次元分しかない)')

## 8. ニューラルネット1層を作ってみる

$$
\mathbf{y} = W \mathbf{x} + \mathbf{b}
$$

In [ ]:
# === 標準形式 ===
# 入力10次元 → 出力5次元のレイヤー
W = rng.standard_normal((5, 10))    # 重み行列
b = rng.standard_normal(5)           # バイアス
x = rng.standard_normal(10)          # 入力

# 線形層
y = W @ x + b
print(f'W.shape = {W.shape}, x.shape = {x.shape}, b.shape = {b.shape}')
print(f'y.shape = {y.shape}')
print(f'y = {y}')

# ReLU活性化
y_relu = np.maximum(y, 0)
print(f'\nReLU適用後: {y_relu}')

In [ ]:
# === ミニバッチ処理 (GPU で並列実行される基本形) ===
BATCH = 32
x_batch = rng.standard_normal((BATCH, 10))    # shape: (32, 10)

# 1サンプルの式: y = W x + b
# バッチの式:    Y = X W^T + b  (転置に注意)
Y_batch = x_batch @ W.T + b
print(f'入力バッチ shape:  {x_batch.shape}')
print(f'出力バッチ shape:  {Y_batch.shape}    ← (32, 5)')

In [ ]:
# === JAX形式 + JITで高速化 ===
import jax

@jax.jit
def linear_layer(W: jnp.ndarray, b: jnp.ndarray, X: jnp.ndarray) -> jnp.ndarray:
    """線形層 + ReLU. shape: (B, in) → (B, out)."""
    return jnp.maximum(X @ W.T + b, 0)

W_j = jnp.array(W)
b_j = jnp.array(b)
X_j = jnp.array(x_batch)

out = linear_layer(W_j, b_j, X_j)
print(f'JAX出力 shape: {out.shape}')
print(out[:3])

## まとめ

ここで触ったもの:
- 行列の作成・shape・スライス
- 単位・対角・対称行列
- 転置・行列積
- 線形変換 (回転)
- 連立方程式の解法
- 行列式・ランク
- ニューラルネット1層 (NumPy + JAX)

## 次へ

次のノート → [`03_eigenvalues.ipynb`](03_eigenvalues.ipynb)

解説 → [`../03_eigenvalues.md`](../03_eigenvalues.md)

---

## 📍 ナビゲーション

| ← 前 | 🏠 章 TOP | 📚 全体 TOP | 次 → |
|---|---|---|---|
| [`01_vectors.ipynb`](01_vectors.ipynb) | [章 TOP](../README.md) | [📚 ROOT README](../../README.md) | [`03_eigenvalues.ipynb`](03_eigenvalues.ipynb) |